# Paper PoRT Prefix Quality Gate Counterfactual Diagnostic

This notebook extends notebook 24. It keeps the same recreated artifact and quality-gate policies, but removes the fallback seed confound by reusing the raw-direct answer/prediction whenever `structure_gate` falls back to the raw prompt.

This is a recreated diagnostic, not an official PoRT paper metric run.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Key difference from notebook 24: `PORT_QUALITY_GATE_REUSE_RAW_FALLBACK=true`.

When a policy falls back to raw prompt, the runner reuses the raw-direct answer and prediction instead of generating again under another seed. This makes the quality-gate comparison a cleaner counterfactual diagnostic.

In [ ]:
os.environ.setdefault('PORT_ARTIFACT_MODE', 'recreated')
os.environ.setdefault('PORT_RUN_NAME', 'paper_port_wmdp_prefix_quality_gate_counterfactual_phi-1_5')
os.environ.setdefault('PORT_MAX_SAMPLES', '32')
os.environ.setdefault('PORT_PREFIX_INCLUDE_BASE_T5', 'false')
os.environ.setdefault('PORT_PREFIX_INCLUDE_RECREATED_T5', 'true')
os.environ.setdefault('PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT', 'true')
os.environ.setdefault('PORT_RECREATED_ARTIFACT_MANIFEST_URL', 'https://raw.githubusercontent.com/toanthangO20/PoRT_LLM_Unlearning-Experiment/artifact-recreated-bootstrap-v1/manifest.json')
os.environ.setdefault('PORT_QUALITY_GATE_INCLUDE_COMPILED_DIRECT', 'true')
os.environ.setdefault('PORT_QUALITY_GATE_INCLUDE_STRUCTURE_GATE', 'true')
os.environ.setdefault('PORT_QUALITY_GATE_INCLUDE_REPAIR_GATE', 'true')
os.environ.setdefault('PORT_QUALITY_GATE_MIN_CHOICE_COVERAGE', '1.0')
os.environ.setdefault('PORT_QUALITY_GATE_MIN_LEN_RATIO', '0.50')
os.environ.setdefault('PORT_QUALITY_GATE_MAX_LEN_RATIO', '2.00')
os.environ.setdefault('PORT_QUALITY_GATE_REQUIRE_PROMPT_INSTRUCTION', 'true')
os.environ.setdefault('PORT_QUALITY_GATE_REQUIRE_ANSWER_INSTRUCTION', 'false')
os.environ.setdefault('PORT_QUALITY_GATE_REPAIR_MAX_PREFIX_CHARS', '600')
os.environ.setdefault('PORT_QUALITY_GATE_REUSE_RAW_FALLBACK', 'true')
os.environ.setdefault('PORT_RESUME_EXISTING', 'true')
os.environ.setdefault('PORT_FAIL_FAST', 'true')
os.environ.setdefault('PORT_BEST_CLASSIFIER_SAMPLES_PER_DOMAIN', '256')
os.environ.setdefault('PORT_BEST_CLASSIFIER_WRONG_ANSWERS_PER_QUESTION', '3')
os.environ.setdefault('PORT_BEST_CLASSIFIER_FEATURE_SET', 'answer_only')
os.environ.setdefault('PORT_BEST_CLASSIFIER_MAX_FEATURES', '50000')

runtime_keys = [
    'PORT_ARTIFACT_MODE',
    'PORT_RUN_NAME',
    'PORT_WMDP_VARIANTS',
    'PORT_WMDP_DOMAINS',
    'PORT_MAX_SAMPLES',
    'PORT_PREFIX_INCLUDE_BASE_T5',
    'PORT_PREFIX_INCLUDE_RECREATED_T5',
    'PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT',
    'PORT_RECREATED_ARTIFACT_MANIFEST_URL',
    'PORT_QUALITY_GATE_INCLUDE_COMPILED_DIRECT',
    'PORT_QUALITY_GATE_INCLUDE_STRUCTURE_GATE',
    'PORT_QUALITY_GATE_INCLUDE_REPAIR_GATE',
    'PORT_QUALITY_GATE_REUSE_RAW_FALLBACK',
    'PORT_QUALITY_GATE_MIN_CHOICE_COVERAGE',
    'PORT_QUALITY_GATE_MIN_LEN_RATIO',
    'PORT_QUALITY_GATE_MAX_LEN_RATIO',
    'PORT_QUALITY_GATE_REQUIRE_PROMPT_INSTRUCTION',
    'PORT_QUALITY_GATE_REQUIRE_ANSWER_INSTRUCTION',
    'PORT_QUALITY_GATE_REPAIR_MAX_PREFIX_CHARS',
    'PORT_RESUME_EXISTING',
    'PORT_FAIL_FAST',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


In [ ]:
import gc
import importlib.util

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('Cleared CUDA cache before run.')
except Exception as exc:
    print('CUDA cache cleanup skipped:', exc)

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_prefix_quality_gate_diagnostic.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_prefix_quality_gate_diagnostic', runner_path)
port_prefix_quality_gate_diagnostic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_prefix_quality_gate_diagnostic)

result = port_prefix_quality_gate_diagnostic.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['run_dir'])
for artifact_name in [
    'artifact_audit.json',
    'run_config.json',
    'summary.json',
    'all_prefix_quality_gate_predictions.csv',
    'prefix_quality_gate_summary_by_job.csv',
    'prefix_quality_gate_summary_overall.csv',
    'failed_jobs.json',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')
